In [1]:
# Setup

required_files <- c(
  "Time for each task - Sheet1.csv",
  "Task Response Form.csv",
  "merged_raw_by_participant.csv"
)

has_required <- function(d) {
  all(file.exists(file.path(d, required_files)))
}

find_base_dir <- function() {
  cwd <- normalizePath(getwd(), winslash = "/", mustWork = FALSE)
  if (has_required(cwd)) return(cwd)

  csv_paths <- list.files(cwd, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE)
  if (length(csv_paths) == 0) {
    stop("Could not find required CSV files.")
  }
  dirs <- unique(dirname(csv_paths))

  # Prefer directory that has both required CSVs and this notebook file.
  nb_name <- "section7_paper_analysis.ipynb"
  for (d in dirs) {
    if (has_required(d) && file.exists(file.path(d, nb_name))) {
      return(normalizePath(d, winslash = "/", mustWork = FALSE))
    }
  }

  # Fallback: first directory that has all required CSVs.
  for (d in dirs) {
    if (has_required(d)) {
      return(normalizePath(d, winslash = "/", mustWork = FALSE))
    }
  }

  stop("Could not find a single directory containing all required CSV files.")
}

find_upward_rlibs <- function(start_dir) {
  cur <- normalizePath(start_dir, winslash = "/", mustWork = FALSE)
  repeat {
    cand <- file.path(cur, ".Rlibs")
    if (dir.exists(cand)) return(cand)
    parent <- dirname(cur)
    if (identical(parent, cur)) break
    cur <- parent
  }
  NULL
}

BASE_DIR <- find_base_dir()
rlibs_dir <- find_upward_rlibs(BASE_DIR)
if (!is.null(rlibs_dir)) {
  .libPaths(c(rlibs_dir, .libPaths()))
}

ensure_pkg <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    user_lib <- Sys.getenv("R_LIBS_USER")
    if (!nzchar(user_lib)) {
      user_lib <- file.path(path.expand("~"), "R", "library")
    }
    dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
    install.packages(pkg, repos = "https://cloud.r-project.org", lib = user_lib)
    .libPaths(c(user_lib, .libPaths()))
  }
  library(pkg, character.only = TRUE)
}

suppressPackageStartupMessages({
  ensure_pkg("lme4")
  ensure_pkg("lmerTest")
})

options(stringsAsFactors = FALSE)

trim_lower <- function(x) tolower(trimws(as.character(x)))
to_num <- function(x) suppressWarnings(as.numeric(trimws(as.character(x))))
to_sec <- function(x) {
  p <- strsplit(trimws(as.character(x)), ":", fixed = TRUE)
  sapply(p, function(v) {
    if (length(v) != 3) return(NA_real_)
    as.numeric(v[1]) * 3600 + as.numeric(v[2]) * 60 + as.numeric(v[3])
  })
}

out_dir <- BASE_DIR
cat("Setup complete.\n")


Setup complete.


## 1) RQ1 Accuracy: Counts + Logistic Mixed Model (Full Fixed Effects)

In [2]:
# Data
acc_csv <- file.path(BASE_DIR, "Time for each task - Sheet1.csv")
d_acc <- read.csv(acc_csv, check.names = FALSE)
d_acc$condition <- trim_lower(d_acc$Interface)
d_acc$correctness <- trim_lower(d_acc$Correctness)
d_acc$is_correct <- ifelse(d_acc$correctness == "correct", 1, 0)

d2 <- subset(
  d_acc,
  condition %in% c("baseline", "hansel") &
    !is.na(is_correct) & !is.na(ID) & !is.na(Task)
)

# Accuracy counts
counts <- aggregate(
  is_correct ~ condition,
  data = d2,
  FUN = function(x) c(n_correct = sum(x), n_total = length(x), accuracy_pct = mean(x) * 100)
)
accuracy_counts <- data.frame(
  condition = counts$condition,
  n_correct = counts$is_correct[, "n_correct"],
  n_total = counts$is_correct[, "n_total"],
  accuracy_pct = counts$is_correct[, "accuracy_pct"]
)
accuracy_counts <- accuracy_counts[order(match(accuracy_counts$condition, c("baseline", "hansel"))), ]

# GLMER

d2$participant <- factor(d2$ID)
d2$task <- factor(d2$Task)
d2$condition_f <- relevel(factor(d2$condition), ref = "baseline")

m_acc <- suppressWarnings(
  glmer(
    is_correct ~ condition_f + task + (1 | participant),
    data = d2,
    family = binomial,
    control = glmerControl(optimizer = "bobyqa", calc.derivs = FALSE)
  )
)

coef_tbl <- as.data.frame(summary(m_acc)$coefficients)
coef_tbl$term <- rownames(coef_tbl)

effect_label <- function(term) {
  if (term == "(Intercept)") return("Task1 @ Baseline (reference)")
  if (term == "condition_fhansel") return("HANSEL vs Baseline")
  if (grepl("^task", term)) return(sprintf("Task%s vs Task1", sub("^task", "", term)))
  term
}

accuracy_glmer_full <- data.frame(
  model = "accuracy ~ condition + task + (1|participant) [glmer binomial]",
  n = nrow(d2),
  participants = nlevels(d2$participant),
  term = coef_tbl$term,
  contrast = vapply(coef_tbl$term, effect_label, character(1)),
  beta = coef_tbl$Estimate,
  se = coef_tbl$`Std. Error`,
  z = coef_tbl$`z value`,
  p = coef_tbl$`Pr(>|z|)`
)

accuracy_counts
accuracy_glmer_full


,condition,n_correct,n_total,accuracy_pct
,<chr>,<dbl>,<dbl>,<dbl>
1,baseline,42,56,75.00000
2,hansel,46,56,82.14286


,model,n,participants,term,contrast,beta,se,z,p
,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,(Intercept),Task1 @ Baseline (reference),3.0485476,1.2922631,2.359076573,0.0183204750
condition_fhansel,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,condition_fhansel,HANSEL vs Baseline,0.7136405,0.6706932,1.064034178,0.2873132166
task2,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,task2,Task2 vs Task1,-2.0947093,1.3797823,-1.518144824,0.1289778934
task3,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,task3,Task3 vs Task1,-1.6061088,1.4149155,-1.135127001,0.2563221100
task4,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,task4,Task4 vs Task1,-0.0257119,1.6751516,-0.015349001,0.9877537496
task5,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,task5,Task5 vs Task1,-0.9575373,1.4801428,-0.646922198,0.5176822981
task6,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,task6,Task6 vs Task1,-5.2323484,1.4710069,-3.556984295,0.0003751366
task7,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,task7,Task7 vs Task1,-0.9575370,1.4801429,-0.646921983,0.5176824373
task8,accuracy ~ condition + task + (1|participant) [glmer binomial],112,14,task8,Task8 vs Task1,16.9990821,5030.0680829,0.003379493,0.9973035595


## 2) RQ2 Time + Effort: LMER (Task1-8 vs) + Wilcoxon

In [3]:
time_csv <- file.path(BASE_DIR, "Time for each task - Sheet1.csv")
resp_csv <- file.path(BASE_DIR, "Task Response Form.csv")

time_raw <- read.csv(time_csv, check.names = FALSE)
resp_raw <- read.csv(resp_csv, check.names = FALSE)

if (!("Participant ID" %in% names(resp_raw))) {
  stop("Task Response Form is missing 'Participant ID' column.")
}

effort_idx <- grep("^How much effort did you need to complete the task\\?$", names(resp_raw))
if (length(effort_idx) != 8) {
  stop(sprintf("Expected 8 effort columns, found %d.", length(effort_idx)))
}

eff_list <- list()
for (i in seq_along(effort_idx)) {
  eff_col <- effort_idx[i]
  eff_list[[i]] <- data.frame(
    ID = to_num(resp_raw[["Participant ID"]]),
    Task = i,
    effort = to_num(resp_raw[[eff_col]])
  )
}
eff_long <- do.call(rbind, eff_list)

trial <- time_raw
trial$ID <- to_num(trial$ID)
trial$Task <- to_num(trial$Task)
trial <- merge(trial, eff_long, by = c("ID", "Task"), all.x = TRUE)

trial$time_sec <- to_sec(trial[["Time diff"]])
trial$condition <- trim_lower(trial$Interface)
trial$correctness <- trim_lower(trial$Correctness)
trial$participant <- factor(trial$ID)
trial$task_f <- relevel(factor(trial$Task), ref = "1")
trial$condition_f <- relevel(factor(trial$condition), ref = "baseline")

# Correct-only for time and effort

d_time <- subset(
  trial,
  correctness == "correct" &
    condition %in% c("baseline", "hansel") &
    !is.na(time_sec) & !is.na(task_f) & !is.na(participant)
)

m_time <- suppressMessages(suppressWarnings(
  lmer(
    time_sec ~ condition_f + task_f + (1 | participant),
    data = d_time,
    REML = TRUE,
    control = lmerControl(optimizer = "bobyqa", calc.derivs = FALSE)
  )
))

time_coef <- as.data.frame(summary(m_time)$coefficients)
time_coef$term <- rownames(time_coef)
get_row <- function(term) time_coef[time_coef$term == term, , drop = FALSE]

extract_effect_row <- function(section, term, contrast_label) {
  x <- get_row(term)
  if (nrow(x) == 0) {
    return(data.frame(
      section = section,
      term = term,
      contrast = contrast_label,
      estimate = NA_real_,
      std_error = NA_real_,
      df = NA_real_,
      t_value = NA_real_,
      p_value = NA_real_
    ))
  }
  data.frame(
    section = section,
    term = term,
    contrast = contrast_label,
    estimate = as.numeric(x$Estimate),
    std_error = as.numeric(x$`Std. Error`),
    df = as.numeric(x$df),
    t_value = as.numeric(x$`t value`),
    p_value = as.numeric(x$`Pr(>|t|)`)
  )
}

# Descriptive means/sd by condition (correct-only)
desc_time <- aggregate(
  time_sec ~ condition,
  data = d_time,
  FUN = function(x) c(mean = mean(x), sd = sd(x), n = length(x))
)
desc_time <- data.frame(
  condition = desc_time$condition,
  mean_sec = desc_time$time_sec[, "mean"],
  sd_sec = desc_time$time_sec[, "sd"],
  n = desc_time$time_sec[, "n"]
)

rows <- list()
rows[[length(rows) + 1]] <- extract_effect_row("condition_effect", "condition_fhansel", "HANSEL vs Baseline")
rows[[length(rows) + 1]] <- data.frame(
  section = "task_effect",
  term = "task_f1_ref",
  contrast = "Task1 vs Task1 (reference)",
  estimate = 0,
  std_error = NA_real_,
  df = NA_real_,
  t_value = NA_real_,
  p_value = NA_real_
)
for (k in 2:8) {
  rows[[length(rows) + 1]] <- extract_effect_row("task_effect", paste0("task_f", k), sprintf("Task%d vs Task1", k))
}

time_lmer_full <- do.call(rbind, rows)
time_lmer_full$analysis <- "correct_only_time_lmer"
time_lmer_full$n_rows <- nrow(d_time)
time_lmer_full$participants <- nlevels(d_time$participant)
time_lmer_full$baseline_mean_sec <- desc_time$mean_sec[desc_time$condition == "baseline"]
time_lmer_full$baseline_sd_sec <- desc_time$sd_sec[desc_time$condition == "baseline"]
time_lmer_full$hansel_mean_sec <- desc_time$mean_sec[desc_time$condition == "hansel"]
time_lmer_full$hansel_sd_sec <- desc_time$sd_sec[desc_time$condition == "hansel"]
time_lmer_full <- time_lmer_full[, c(
  "analysis", "section", "term", "contrast",
  "estimate", "std_error", "df", "t_value", "p_value",
  "n_rows", "participants",
  "baseline_mean_sec", "baseline_sd_sec", "hansel_mean_sec", "hansel_sd_sec"
)]

# Effort Wilcoxon on participant-level medians (correct-only)

d_eff <- subset(
  trial,
  correctness == "correct" &
    condition %in% c("baseline", "hansel") &
    !is.na(effort)
)

agg_eff <- aggregate(effort ~ participant + condition, data = d_eff, FUN = median)
wide <- reshape(agg_eff, idvar = "participant", timevar = "condition", direction = "wide")
names(wide) <- sub("^effort\\.", "", names(wide))
wide <- wide[!is.na(wide$baseline) & !is.na(wide$hansel), ]

wt <- suppressWarnings(
  wilcox.test(
    wide$hansel, wide$baseline,
    paired = TRUE,
    alternative = "two.sided",
    exact = FALSE,
    correct = FALSE
  )
)

n_pairs <- nrow(wide)
p_w <- wt$p.value
z_abs <- qnorm(p_w / 2, lower.tail = FALSE)
z_signed <- sign(median(wide$hansel - wide$baseline)) * z_abs
r_abs <- abs(z_signed) / sqrt(n_pairs)

effort_wilcoxon <- data.frame(
  analysis = "correct_only_effort_wilcoxon_participant_medians",
  n_rows = nrow(d_eff),
  n_pairs = n_pairs,
  V = as.numeric(wt$statistic),
  p_value = p_w,
  effect_r_abs = r_abs,
  baseline_mean_of_participant_medians = mean(wide$baseline),
  hansel_mean_of_participant_medians = mean(wide$hansel),
  baseline_median_of_participant_medians = median(wide$baseline),
  hansel_median_of_participant_medians = median(wide$hansel)
)

time_lmer_full
effort_wilcoxon


analysis,section,term,contrast,estimate,std_error,df,t_value,p_value,n_rows,participants,baseline_mean_sec,baseline_sd_sec,hansel_mean_sec,hansel_sd_sec
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
correct_only_time_lmer,condition_effect,condition_fhansel,HANSEL vs Baseline,-43.199156,10.03239,65.85427,-4.3059689,5.652775e-05,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f1_ref,Task1 vs Task1 (reference),0.000000,NA,NA,NA,NA,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f2,Task2 vs Task1,14.187394,19.27419,65.27246,0.7360825,4.643176e-01,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f3,Task3 vs Task1,-50.855325,18.71697,64.83817,-2.7170705,8.436592e-03,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f4,Task4 vs Task1,4.292736,17.95428,65.22522,0.2390926,8.117835e-01,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f5,Task5 vs Task1,15.772590,18.24777,64.40679,0.8643573,3.905994e-01,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f6,Task6 vs Task1,23.650220,30.45990,68.43958,0.7764377,4.401657e-01,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f7,Task7 vs Task1,-53.340817,18.40147,65.76361,-2.8987255,5.086941e-03,88,14,167.6905,60.9876,130.3913,53.2017
correct_only_time_lmer,task_effect,task_f8,Task8 vs Task1,-42.733312,17.56875,64.70836,-2.4323473,1.777667e-02,88,14,167.6905,60.9876,130.3913,53.2017


analysis,n_rows,n_pairs,V,p_value,effect_r_abs,baseline_mean_of_participant_medians,hansel_mean_of_participant_medians,baseline_median_of_participant_medians,hansel_median_of_participant_medians
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
correct_only_effort_wilcoxon_participant_medians,88,14,7,0.01170302,0.6737589,4.392857,2.892857,4.5,2.75


## 3) RQ3 Usefulness: Preference + Paired Ratings + Feature Summary

In [4]:
rating_csv <- file.path(BASE_DIR, "merged_raw_by_participant.csv")
d <- read.csv(rating_csv, check.names = FALSE)

prefix <- "post_study_survey_hansel_first_form_responses_1__"
col_group <- "schedule_of_study_sheet1__Group"
col_pref_overall <- paste0(prefix, "Which interface did you prefer overall?")
col_pref_verify <- paste0(prefix, "Which interface made it easier for you to verify whether the agent’s answer was correct?")

ab_items <- list(
  list(
    key = "understanding",
    label = "Understanding",
    hansel_col = paste0(prefix, "The information provided by the interface helped me understand how the agent arrived at the answer."),
    baseline_col = paste0(prefix, "The information provided by the interface helped me understand how the agent arrived at the answer.__2")
  ),
  list(
    key = "verification",
    label = "Verification",
    hansel_col = paste0(prefix, "The information provided by the interface made it easy to verify the agent’s response."),
    baseline_col = paste0(prefix, "The information provided by the interface made it easy to verify the agent’s response.__2")
  ),
  list(
    key = "error_identification",
    label = "Error Identification",
    hansel_col = paste0(prefix, "The information provided by the interface helped me identify potential problems in the agent’s response (e.g., using incorrect information or missing important sorting steps)."),
    baseline_col = paste0(prefix, "The information provided by the interface helped me identify where the agent went wrong (e.g., using incorrect information or missing important steps)")
  ),
  list(
    key = "correction",
    label = "Correction",
    hansel_col = paste0(prefix, "When the agent’s response was wrong, the information provided by the interface helped me revise it."),
    baseline_col = paste0(prefix, "When the agent’s response was wrong, the information provided by the interface helped me revise it.__2")
  ),
  list(
    key = "usability",
    label = "Usability",
    hansel_col = paste0(prefix, "The interface was easy to learn and use."),
    baseline_col = paste0(prefix, "The interface was easy to learn and use.__2")
  )
)

feature_items <- list(
  list(
    key = "highlighted_evidence",
    label = "Highlighted Evidence",
    col = paste0(prefix, "For interface A, seeing highlighted information on the pages (e.g., \"Global Gas Station\", \"4.3\")")
  ),
  list(
    key = "direct_interaction_pages",
    label = "Direct Interaction with Pages",
    col = paste0(prefix, "For interface A, interacting with the evidence pages directly (e.g., clicking, scrolling)")
  ),
  list(
    key = "evidence_pages_titles",
    label = "Evidence Pages with Titles",
    col = paste0(prefix, "For interface A, viewing the evidence pages with short title")
  ),
  list(
    key = "source_links",
    label = "Source Links",
    col = paste0(prefix, "For interface B, viewing the source links")
  ),
  list(
    key = "step_by_step_trajectory",
    label = "Step-by-step Trajectory",
    col = paste0(prefix, "For both interface A and B, viewing the step-by-step trajectory (e.g., show 22 steps)")
  )
)

# Preference mapped to HANSEL
first_letter <- function(x) {
  out <- sub("^\\s*([AB]).*$", "\\1", trimws(as.character(x)))
  out[!out %in% c("A", "B")] <- NA_character_
  out
}

group <- trimws(as.character(d[[col_group]]))
overall_pick <- first_letter(d[[col_pref_overall]])
verify_pick <- first_letter(d[[col_pref_verify]])
valid_group <- group %in% c("A", "B")

preference_hansel_counts <- data.frame(
  question = c("overall_preference", "verification_ease_preference"),
  n_total = c(sum(valid_group & !is.na(overall_pick)), sum(valid_group & !is.na(verify_pick))),
  n_prefer_hansel = c(
    sum(valid_group & !is.na(overall_pick) & overall_pick == group),
    sum(valid_group & !is.na(verify_pick) & verify_pick == group)
  )
)
preference_hansel_counts$pct_prefer_hansel <- preference_hansel_counts$n_prefer_hansel / preference_hansel_counts$n_total * 100

# Paired tests by dimension
ab_rows <- list()
for (it in ab_items) {
  h <- to_num(d[[it$hansel_col]])
  b <- to_num(d[[it$baseline_col]])
  ok <- is.finite(h) & is.finite(b)
  h <- h[ok]
  b <- b[ok]
  n <- length(h)
  if (n == 0) next

  wt <- suppressWarnings(wilcox.test(h, b, paired = TRUE, alternative = "two.sided", exact = FALSE, correct = FALSE))
  p_w <- wt$p.value
  z_abs <- qnorm(p_w / 2, lower.tail = FALSE)
  z_signed <- sign(median(h - b)) * z_abs
  r_abs <- abs(z_signed) / sqrt(n)

  ab_rows[[length(ab_rows) + 1]] <- data.frame(
    item_key = it$key,
    item_label = it$label,
    n_pairs = n,
    mean_baseline = mean(b),
    mean_hansel = mean(h),
    median_baseline = median(b),
    median_hansel = median(h),
    mean_diff_hansel_minus_baseline = mean(h - b),
    wilcoxon_V = as.numeric(wt$statistic),
    p_wilcoxon = p_w,
    effect_r_abs = r_abs
  )
}
rating_hansel_vs_baseline <- do.call(rbind, ab_rows)

# Feature summary
feat_rows <- list()
for (it in feature_items) {
  x <- to_num(d[[it$col]])
  x <- x[is.finite(x)]
  if (length(x) == 0) next
  feat_rows[[length(feat_rows) + 1]] <- data.frame(
    feature_key = it$key,
    feature_label = it$label,
    n = length(x),
    mean_score = mean(x),
    n_ge4 = sum(x >= 4),
    pct_ge4 = mean(x >= 4) * 100
  )
}
feature_summary <- do.call(rbind, feat_rows)
feature_summary <- feature_summary[order(-feature_summary$mean_score), ]

preference_hansel_counts
rating_hansel_vs_baseline
feature_summary


question,n_total,n_prefer_hansel,pct_prefer_hansel
<chr>,<int>,<int>,<dbl>
overall_preference,14,14,100
verification_ease_preference,14,14,100


item_key,item_label,n_pairs,mean_baseline,mean_hansel,median_baseline,median_hansel,mean_diff_hansel_minus_baseline,wilcoxon_V,p_wilcoxon,effect_r_abs
<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
understanding,Understanding,14,3.071429,4.500000,3.0,5.0,1.428571,78.0,0.001725471,0.8375484
verification,Verification,14,2.428571,4.428571,2.5,4.5,2.000000,91.0,0.001335105,0.8574609
error_identification,Error Identification,14,2.571429,4.000000,2.5,4.0,1.428571,60.5,0.012738072,0.6657503
correction,Correction,14,2.500000,3.928571,2.0,4.0,1.428571,55.0,0.004481980,0.7595787
usability,Usability,14,2.928571,4.714286,3.0,5.0,1.785714,55.0,0.004455352,0.7600862


,feature_key,feature_label,n,mean_score,n_ge4,pct_ge4
,<chr>,<chr>,<int>,<dbl>,<int>,<dbl>
1,highlighted_evidence,Highlighted Evidence,14,4.785714,13,92.85714
2,direct_interaction_pages,Direct Interaction with Pages,14,4.642857,14,100.00000
3,evidence_pages_titles,Evidence Pages with Titles,14,4.428571,14,100.00000
5,step_by_step_trajectory,Step-by-step Trajectory,14,3.357143,9,64.28571
4,source_links,Source Links,14,3.142857,6,42.85714
